In [ ]:
import numpy as np
import torch

from src.rl.env_autoscale import AutoscaleEnv
from src.rl.agent_ppo import PPOAgent


In [ ]:
agent = PPOAgent(state_dim=4, action_dim=3)
agent.policy.load_state_dict(torch.load("models/rl_policy.pt"))
agent.value.load_state_dict(torch.load("models/rl_value.pt"))


In [ ]:
def heuristic_action(state):
    cpu, requests_norm, latency_norm, replicas_norm = state
    # Simple rule:
    # if cpu > 0.8 -> scale up
    # if cpu < 0.4 -> scale down
    # else hold
    if cpu > 0.8:
        return 2  # scale_up
    elif cpu < 0.4:
        return 0  # scale_down
    else:
        return 1  # hold


In [ ]:
def eval_rl(episodes=50):
    env = AutoscaleEnv()
    returns = []

    for _ in range(episodes):
        state = env.reset()
        done = False
        ep_return = 0

        while not done:
            state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            logits = agent.policy(state_t)
            probs = torch.softmax(logits, dim=-1)
            action = torch.argmax(probs, dim=-1).item()

            next_state, reward, done, _ = env.step(action)
            ep_return += reward
            state = next_state

        returns.append(ep_return)

    return np.mean(returns), returns


In [ ]:
def eval_heuristic(episodes=50):
    env = AutoscaleEnv()
    returns = []

    for _ in range(episodes):
        state = env.reset()
        done = False
        ep_return = 0

        while not done:
            action = heuristic_action(state)
            next_state, reward, done, _ = env.step(action)
            ep_return += reward
            state = next_state

        returns.append(ep_return)

    return np.mean(returns), returns


In [ ]:
mean_rl, returns_rl = eval_rl(episodes=50)
mean_heur, returns_heur = eval_heuristic(episodes=50)

print("RL average return:", mean_rl)
print("Heuristic average return:", mean_heur)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(returns_rl, label="RL")
plt.plot(returns_heur, label="Heuristic")
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title("RL vs Heuristic Autoscaling")
plt.legend()
plt.grid(True)
plt.show()
